## Location-Based Listing Recommendation

In this notebook, I use the final Random Forest model to recommend which listing types might perform best for a specific latitude and longitude.

The goal is to move from general model results into a more practical recommendation tool. Instead of only asking which features matter overall, I want to compare nearby listings and test different listing scenarios in the same location.

### Steps

- Load the final product modeling dataset.
- Load the saved Random Forest model and feature list.
- Choose a target latitude and longitude.
- Calculate each listing’s distance from that point.
- Filter nearby listings within a selected radius.
- Review the strongest local comps based on `reviews_per_month`.
- Build a few hypothetical listing scenarios for the same location.
- Use the model to predict demand for each scenario.
- Rank the scenarios and compare the model output to real local comps.

In [1]:
!pip install geopy

In [2]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from geopy.distance import geodesic

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

sns.set_theme(style='whitegrid')

In [3]:
# load final product feature dataset and saved recommendation model
df = pd.read_csv('../data/processed/airbnb_product_modeling.csv')

rf_model = joblib.load('../models/rf_location_recommendation_model.pkl')
rf_features = joblib.load('../models/rf_location_recommendation_features.pkl')

print('Recommendation dataset shape:', df.shape)
print('Model feature count:', len(rf_features))

# checking that the dataset has all features the model expects
missing_features = [col for col in rf_features if col not in df.columns]

print('Missing model features:', missing_features)

Recommendation dataset shape: (210525, 102)
Model feature count: 14
Missing model features: []


### Target location

I am starting with a test location in downtown San Diego. The same workflow can be reused with any latitude and longitude.

In [4]:
# checking San Diego listings only
df[df['city'] == 'San Diego'][
    ['id', 'city', 'latitude', 'longitude', 'reviews_per_month']].head()

,id,city,latitude,longitude,reviews_per_month
177271,610920,San Diego,32.81244,-117.26835,0.13
177272,6,San Diego,32.75522,-117.12873,0.85
177273,29967,San Diego,32.80751,-117.25760,0.59
177274,1166766,San Diego,32.78337,-117.19911,0.22
177275,1167130,San Diego,32.74569,-117.18702,0.24


In [5]:
# setting target location
target_lat = 32.7157
target_lon = -117.1611
target_point = (target_lat, target_lon)

# calculating distance from the target location
df['distance_miles'] = df.apply(
    lambda row: geodesic(
        target_point,
        (row['latitude'], row['longitude'])
    ).miles,
    axis=1)

# checking closest listings to the target point
df[
    ['id', 'city', 'latitude', 'longitude', 'distance_miles', 'reviews_per_month']
].sort_values('distance_miles').head(10)

,id,city,latitude,longitude,distance_miles,reviews_per_month
184322,52519726,San Diego,32.714910,-117.160830,0.056665,2.84
187959,808033005728294337,San Diego,32.714891,-117.160732,0.059701,41.45
186986,699131140412566866,San Diego,32.714790,-117.160960,0.063236,0.67
181522,37866535,San Diego,32.714890,-117.160550,0.064359,0.19
187344,740113564630839614,San Diego,32.714980,-117.160360,0.065727,2.76
184449,52861672,San Diego,32.714890,-117.160500,0.065858,1.55
184776,53798122,San Diego,32.714830,-117.160520,0.068817,3.36
185735,611024172096437999,San Diego,32.714890,-117.160370,0.070171,1.55
182895,46669557,San Diego,32.714890,-117.160330,0.071607,3.32
185873,622507980626969677,San Diego,32.714940,-117.160130,0.077044,2.57


In [6]:
# filtering nearby listings within the target radius
radius_miles = 3

local_comps = df[df['distance_miles'] <= radius_miles].copy()

print('Local comps shape:', local_comps.shape)

local_comps[
    ['id', 'city', 'distance_miles', 'reviews_per_month',
     'room_type', 'accommodates', 'bedrooms', 'bathrooms',
     'price_clean', 'family_friendly_score', 'work_friendly_score',
     'amenity_strength_score', 'pricing_competitiveness_score']
].sort_values('reviews_per_month', ascending=False).head(10)

Local comps shape: (3167, 103)


,id,city,distance_miles,reviews_per_month,room_type,accommodates,bedrooms,bathrooms,price_clean,family_friendly_score,work_friendly_score,amenity_strength_score,pricing_competitiveness_score
187878,808814447273523115,San Diego,0.084413,48.82,Entire home/apt,2,1.0,1.0,110.0,0,63,38,88.000000
187877,808798275080007348,San Diego,0.221334,45.27,Entire home/apt,2,1.0,1.0,132.0,0,63,38,94.400000
187959,808033005728294337,San Diego,0.059701,41.45,Entire home/apt,2,1.0,1.0,124.0,0,63,38,99.200000
187738,787197038845726397,San Diego,0.264672,34.43,Private room,2,1.0,5.0,45.0,15,10,38,60.000000
187881,808859465062374658,San Diego,0.180584,31.09,Entire home/apt,2,1.0,1.0,119.0,0,63,38,95.200000
180554,29278867,San Diego,2.124313,13.07,Entire home/apt,2,1.0,1.0,161.0,10,53,57,71.200000
180353,26468774,San Diego,1.789695,11.34,Entire home/apt,2,1.0,1.0,123.0,25,35,65,98.400000
181629,38902052,San Diego,0.570018,11.04,Entire home/apt,2,0.0,1.0,669.0,15,25,55,0.000000
188082,832989842561439647,San Diego,0.768572,11.00,Entire home/apt,4,1.0,1.0,158.0,100,75,85,96.932515
183799,50697480,San Diego,0.640205,10.87,Entire home/apt,4,1.0,1.0,147.0,75,73,78,90.184049
